# Projeto e Análise de Algoritmos — Lista 2 (Extra)
### Grafos e Classes de Complexidade — FGV - EMAp — 2026

Notebook-template para organização das soluções. Cada questão possui:

1. Uma célula **markdown** com o enunciado (extraído do PDF, com notação em LaTeX);
2. Uma célula **code** com o template original da função (extraído de `lista-2-paa.py`);
3. Uma célula **code** com os exemplos do enunciado para testar a sua solução (quando se aplica).

> **Atenção:** ao final, o conteúdo de todas as células de código das funções deve ser copiado de volta para o arquivo `lista-2-paa.py`, mantendo nomes de arquivo e assinaturas de funções exatamente como estão. Código auxiliar (classes, funções) deve ser definido **dentro** do escopo da função da questão correspondente.
>
> O limite de recursão padrão do Python é pequeno (~1000) e várias entradas são grandes o bastante para estourá-lo. Nas questões de análise de complexidade, justifique o custo do que você implementou.

---
## 1. Alfabeto Perdido

Arqueólogos encontraram uma lista de palavras de uma língua desconhecida. Eles sabem que a lista está em ordem lexicográfica segundo a ordem alfabética dessa língua, mas não sabem qual é essa ordem.

Dada a lista de palavras, retorne uma string contendo cada letra distinta que aparece nas palavras exatamente uma vez, em uma ordem consistente com a ordenação observada. Se existir mais de uma ordem válida, qualquer uma é aceita. Se a lista for inconsistente e nenhuma ordem de letras a explicaria, retorne a string vazia `""`. Uma palavra nunca pode vir antes de um prefixo próprio seu.

Seu algoritmo deve executar em $O(C)$, onde $C$ é o total de caracteres da lista.

**Restrições:**
- o alfabeto tem no máximo 26 letras
- $1 \le$ número de palavras $\le 10^5$
- Palavras com letras de `a` a `z`, não vazias.
- A lista pode conter palavras repetidas.

**Exemplos:**

| Entrada | Saída |
|---|---|
| `["zx", "zy", "yx"]` | `"zxy"` ou `"xzy"` |
| `["cab", "cad", "bd", "bc"]` | `""` |
| `["abc", "ab"]` | `""` |
| `["abc"]` | qualquer permutação de `"abc"` |

In [5]:
def question_1(palavras: list) -> str:
    """
    Alfabeto Perdido.

    palavras: lista de palavras (letras 'a'..'z', não vazias) em ordem
              lexicográfica segundo uma ordem de letras DESCONHECIDA.
              1 <= len(palavras) <= 10^5; o alfabeto tem no máximo 26 letras;
              a lista pode conter palavras repetidas.

    Retorna uma string com cada letra distinta que aparece nas palavras
    exatamente uma vez, em uma ordem consistente com a ordenação observada
    (qualquer ordem válida é aceita). Se a lista for inconsistente, retorna "".
    Uma palavra nunca pode vir antes de um prefixo próprio seu
    (ex: "abc" antes de "ab" é inconsistente).

    Complexidade exigida: O(C), C = total de caracteres da lista.
    """

    class CharNode:
        def __init__(self, label:str) -> None:
            self.label = label
            self.in_edges = list()
            self.out_edges = list()

        def add_in_edge(self, edge:CharEdge) -> None:
            self.in_edges.append(edge)
            
        def add_out_edge(self, edge:CharEdge) -> None:
            self.out_edges.append(edge)

    class CharEdge:
        def __init__(self, origem:CharNode, destino:CharNode) -> None:
            self.origem = origem
            self.destino = destino

            origem.add_out_edge(self)
            destino.add_in_edge(self)

    def ordem_topologica(letras: list[CharNode]) -> list[CharNode]:
        n = len(letras)
        in_deg = {l: len(l.in_edges) for l in letras}

        fila = [l for l, d in in_deg.items() if d == 0]
        atual = 0
        while atual < n:
            if atual == len(fila):
                return list()

            for e in fila[atual].out_edges:
                in_deg[e.destino] -= 1
                if in_deg[e.destino] == 0:
                    fila.append(e.destino)

            atual += 1

        return fila

    n = len(palavras)
    if n == 0:
        return ''
    
    vertices = dict()
    for c in palavras[0]:
        if c not in vertices:
            vertices[c] = CharNode(c)

    tam_atual = len(palavras[0])
    tam_anterior = tam_atual
    for i in range(1, n):
        tam_atual = len(palavras[i])
        for j in range(tam_atual):
            letra_atual = palavras[i][j]
            if j >= tam_anterior:
                if letra_atual not in vertices:
                    vertices[letra_atual] = CharNode(letra_atual)
                    break
            if letra_atual == palavras[i-1][j]:
                if j == tam_atual - 1:
                    return ''
                else:
                    continue

            # caso tenha uma nova letra (deve vir após palavras[i-1][j])
            if letra_atual not in vertices:
                vertices[letra_atual] = CharNode(letra_atual)
            CharEdge(vertices[palavras[i-1][j]], vertices[letra_atual]) 
            break

        tam_anterior = tam_atual

    ordem = ordem_topologica([v for v in vertices.values()])
    ordem_str = ''
    for v in ordem:
        ordem_str += v.label
    return ordem_str

In [6]:
from itertools import permutations

entradas = [
    (["zx", "zy", "yx"],),
    (["cab", "cad", "bd", "bc"],),
    (["abc", "ab"],),
    (["abc"],)
]
saidas = [
    ["zxy", "xzy"],
    [""],
    [""],
    ["".join(p) for p in permutations("abc")]
]  # lista de saídas aceitas para cada entrada
sucessos = 0

print("="*5 + " Exemplos Apresentados " + "="*5 )

for entrada, saida in zip(entradas, saidas):
    print()

    print("Entrada: ", entrada)
    print("Resultado Esperado: ", saida)

    resultado = question_1(*entrada)
    print("Resultado Obtido: ", resultado)

    sucesso = resultado in saida
    print("Sucesso: ", sucesso)
    if sucesso: sucessos += 1

    print()
    print("="*34)

print()
print("Total de Sucessos: \t", sucessos)
print("Total de Falhas: \t", len(saidas) - sucessos)

===== Exemplos Apresentados =====

Entrada:  (['zx', 'zy', 'yx'],)
Resultado Esperado:  ['zxy', 'xzy']
Resultado Obtido:  zxy
Sucesso:  True


Entrada:  (['cab', 'cad', 'bd', 'bc'],)
Resultado Esperado:  ['']
Resultado Obtido:  
Sucesso:  True


Entrada:  (['abc', 'ab'],)
Resultado Esperado:  ['']
Resultado Obtido:  
Sucesso:  True


Entrada:  (['abc'],)
Resultado Esperado:  ['abc', 'acb', 'bac', 'bca', 'cab', 'cba']
Resultado Obtido:  abc
Sucesso:  True


Total de Sucessos: 	 4
Total de Falhas: 	 0


---
## 2. Ponte entre as Ilhas

Um mapa é representado por uma matriz de 0s (água) e 1s (terra). Uma ilha é um conjunto maximal de células de terra conectadas na vertical ou na horizontal. O mapa contém exatamente duas ilhas.

O governo quer ligar as duas ilhas com uma ponte, convertendo células de água em terra. Retorne o menor número de células de água que precisam ser convertidas para que as duas ilhas passem a formar uma só.

Seu algoritmo deve executar em $O(n \cdot m)$.

**Restrições:**
- $2 \le n, m \le 500$
- Há exatamente duas ilhas.

**Exemplos:**

| Entrada | Saída |
|---|---|
| `[[1,1,0,0,0],[1,0,0,0,0],[0,0,0,0,0],[0,0,0,1,1],[0,0,0,1,1]]` | 4 |
| `[[1,0,0,0,1],[1,0,0,0,1],[1,0,0,0,1]]` | 3 |
| `[[0,1,0],[0,0,0],[0,0,1]]` | 2 |

In [ ]:
def question_2(mapa: list) -> int:
    """
    Ponte entre as Ilhas.

    mapa: matriz n x m (lista de listas) de 0 (água) e 1 (terra), com
          2 <= n, m <= 500. Há exatamente duas ilhas (conjuntos maximais de
          células de terra conectadas na vertical/horizontal).

    Retorna o menor número de células de água que precisam ser convertidas
    em terra para que as duas ilhas formem uma só.

    Complexidade exigida: O(n · m).
    """
    pass

In [ ]:
entradas = [
    ([[1, 1, 0, 0, 0],
      [1, 0, 0, 0, 0],
      [0, 0, 0, 0, 0],
      [0, 0, 0, 1, 1],
      [0, 0, 0, 1, 1]],),
    ([[1, 0, 0, 0, 1],
      [1, 0, 0, 0, 1],
      [1, 0, 0, 0, 1]],),
    ([[0, 1, 0],
      [0, 0, 0],
      [0, 0, 1]],)
]
saidas = [
    4,
    3,
    2
]
sucessos = 0

print("="*5 + " Exemplos Apresentados " + "="*5 )

for entrada, saida in zip(entradas, saidas):
    print()

    print("Entrada: ", entrada)
    print("Resultado Esperado: ", saida)

    resultado = question_2(*entrada)
    print("Resultado Obtido: ", resultado)

    sucesso = resultado == saida
    print("Sucesso: ", sucesso)
    if sucesso: sucessos += 1

    print()
    print("="*34)

print()
print("Total de Sucessos: \t", sucessos)
print("Total de Falhas: \t", len(saidas) - sucessos)

---
## 3. Trilha de Menor Esforço

Um mapa topográfico é uma matriz $n \times m$ de altitudes. Uma caminhante parte da célula superior esquerda $(0, 0)$ e quer chegar à inferior direita $(n-1, m-1)$, movendo-se a cada passo para uma das quatro células vizinhas (cima, baixo, esquerda, direita).

O esforço de uma trilha é a maior diferença absoluta de altitude entre duas células consecutivas ao longo dela.

Retorne uma tupla `(esforço, passos)`, em que `esforço` é o menor esforço possível entre todas as trilhas, e `passos` é o menor número de passos entre as trilhas que atingem esse esforço mínimo.

Seu algoritmo deve executar em $O(n \cdot m \cdot \log(n \cdot m))$ ou melhor.

**Restrições:**
- $1 \le n, m \le 300$
- $1 \le \text{altitude} \le 10^6$

**Exemplos:**

| Entrada | Saída |
|---|---|
| `[[4,3,4],[3,3,3],[4,3,4]]` | `(1, 4)` |
| `[[1,1,1]]` | `(0, 2)` |
| `[[1],[5],[1]]` | `(4, 2)` |

In [ ]:
def question_3(altitudes: list) -> tuple:
    """
    Trilha de Menor Esforço.

    altitudes: matriz n x m (lista de listas) de inteiros,
               1 <= n, m <= 300, 1 <= altitude <= 10^6.
    A caminhante parte de (0, 0) e chega em (n-1, m-1), movendo-se para uma das
    4 células vizinhas a cada passo. O esforço de uma trilha é a maior
    diferença absoluta de altitude entre duas células consecutivas dela.

    Retorna (esforco, passos): o menor esforço possível entre todas as trilhas
    e, entre as trilhas que atingem esse esforço, o menor número de passos.

    Complexidade exigida: O(n · m · log(n · m)) ou melhor.
    """
    # Resolvido via Dijkstra com min heap de caminhos. 
    # Graças à min heap, encontrar e popar a aresta que completa o menor caminho da iteração é 
    # feito em O(log |E|), de modo que o algoritmo funcione em O((|V|+|E|)*log|E|).
    # Dado que |E| <= 4*|V|, por ser uma grid, O((|V|+|E|)*log|E|) = O(|V|*log|V|). 

    class ArestaNode:
        def __init__(self, origem:tuple, destino:tuple, custo_usando:int) -> None:
            self.custo = custo_usando
            self.origem = origem
            self.destino = destino

        def id(self):
            return self.custo

    class HeapMin:
        def __init__(self) -> None:
            self.len = 0
            self.heap = list()

        def push(self, node: ArestaNode) -> None:
            self.heap.append(node)
            atual = self.len 
            next = int((atual - 1)/2)
            while next > 0:
                if (self.heap[atual].id() < self.heap[next].id()):
                    self.heap[atual], self.heap[next] = self.heap[next], self.heap[atual]
                else:
                    break

                atual = next
                next = int((atual-1)/2)

            self.len += 1

        def pop(self) -> ArestaNode | None:
            if (self.len < 1):
                return None

            self.len -= 1
            self.heap[0], self.heap[self.len] = self.heap[self.len], self.heap[0]
            if (self.len <= 1):
                return self.heap.pop()
           
            atual = 0
            while 2*atual+1 < self.len:
                if (2*atual+2 < self.len):
                    if (self.heap[2*atual+2].id() < self.heap[2*atual+1].id()):
                        min_child_pos = 2*atual+2
                    else:
                        min_child_pos = 2*atual+1
                else:
                    min_child_pos = 2*atual+1

                if (self.heap[atual].id() > self.heap[min_child_pos].id()):
                    self.heap[atual], self.heap[min_child_pos] = self.heap[min_child_pos], self.heap[atual]
                else:
                    break

                atual = min_child_pos
            
            return self.heap.pop()

    visitados = {(0,0): (0,0)} # vértice de origem: melhor par <esforço, passos> até ele
    arestas = HeapMin()

    n = len(altitudes)
    if n > 1:
        arestas.push(ArestaNode((0,0), (1,0), abs(altitudes[1][0] - altitudes[0][0])))
    m = len(altitudes[0])
    if m > 1:
        arestas.push(ArestaNode((0,0), (0,1), abs(altitudes[0][1] - altitudes[0][0])))
    
    while arestas.len > 0:
        aresta_min = arestas.pop()
        if aresta_min == None:
            # nunca deve cair aqui pq len > 0, só pro pylance parar de reclamar
            continue
        if aresta_min.destino not in visitados:
            visitados[aresta_min.destino] = aresta_min.custo, visitados[aresta_min.origem][1]+1
        elif ((aresta_min.custo < visitados[aresta_min.origem][0]) or
              (aresta_min.custo == visitados[aresta_min.origem][0] and
               visitados[aresta_min.origem][1]+1 < visitados[aresta_min.destino][1])):
            visitados[aresta_min.destino] = aresta_min.custo, visitados[aresta_min.origem][1]+1
        else:
            continue # esse caminho é pior que o usado anteriormente

        # TODO: não tem necessidade de adicionar todas, só as que levam a alguém não visitado ainda 
        # por definição, se já está visitado, aquele o caminho de menor custo já foi percorrido
        if ((aresta_min.destino[0] > 0) and ((aresta_min.destino[0]-1, aresta_min.destino[1]) != aresta_min.origem)):
            next = (aresta_min.destino[0]-1, aresta_min.destino[1])
            arestas.push(ArestaNode(aresta_min.destino, next, 
                max(visitados[aresta_min.destino][0],abs(altitudes[next[0]][next[1]] - altitudes[aresta_min.destino[0]][aresta_min.destino[1]]))))
        
        if ((aresta_min.destino[0] < n-1) and ((aresta_min.destino[0]+1, aresta_min.destino[1]) != aresta_min.origem)):
            next = (aresta_min.destino[0]+1, aresta_min.destino[1])
            arestas.push(ArestaNode(aresta_min.destino, next, 
                max(visitados[aresta_min.destino][0],abs(altitudes[next[0]][next[1]] - altitudes[aresta_min.destino[0]][aresta_min.destino[1]]))))
        
        if ((aresta_min.destino[1] > 0) and ((aresta_min.destino[0], aresta_min.destino[1]-1) != aresta_min.origem)):
            next = (aresta_min.destino[0], aresta_min.destino[1]-1)
            arestas.push(ArestaNode(aresta_min.destino, next, 
                max(visitados[aresta_min.destino][0],abs(altitudes[next[0]][next[1]] - altitudes[aresta_min.destino[0]][aresta_min.destino[1]]))))
        
        if ((aresta_min.destino[1] < m-1) and ((aresta_min.destino[0], aresta_min.destino[1]+1) != aresta_min.origem)):
            next = (aresta_min.destino[0], aresta_min.destino[1]+1)
            arestas.push(ArestaNode(aresta_min.destino, next, 
                max(visitados[aresta_min.destino][0],abs(altitudes[next[0]][next[1]] - altitudes[aresta_min.destino[0]][aresta_min.destino[1]]))))
            
    return visitados[(n-1, m-1)] 

In [25]:
entradas = [
    ([[4, 3, 4], [3, 3, 3], [4, 3, 4]],),
    ([[1, 1, 1]],),
    ([[1], [5], [1]],)
]
saidas = [
    (1, 4),
    (0, 2),
    (4, 2)
]
sucessos = 0

print("="*5 + " Exemplos Apresentados " + "="*5 )

for entrada, saida in zip(entradas, saidas):
    print()

    print("Entrada: ", entrada)
    print("Resultado Esperado: ", saida)

    resultado = question_3(*entrada)
    print("Resultado Obtido: ", resultado)

    sucesso = resultado == saida
    print("Sucesso: ", sucesso)
    if sucesso: sucessos += 1

    print()
    print("="*34)

print()
print("Total de Sucessos: \t", sucessos)
print("Total de Falhas: \t", len(saidas) - sucessos)

===== Exemplos Apresentados =====

Entrada:  ([[4, 3, 4], [3, 3, 3], [4, 3, 4]],)
Resultado Esperado:  (1, 4)
Resultado Obtido:  (1, 4)
Sucesso:  True


Entrada:  ([[1, 1, 1]],)
Resultado Esperado:  (0, 2)
Resultado Obtido:  (0, 2)
Sucesso:  True


Entrada:  ([[1], [5], [1]],)
Resultado Esperado:  (4, 2)
Resultado Obtido:  (4, 2)
Sucesso:  True


Total de Sucessos: 	 3
Total de Falhas: 	 0


---
## 4. Câmbio Ineficiente

Uma casa de câmbio opera com $n$ moedas, numeradas de $0$ a $n-1$. Uma cotação $(a, b, r)$, com $a \neq b$, significa que 1 unidade da moeda $a$ pode ser trocada por $r$ unidades da moeda $b$ (a cotação é só nesse sentido; a troca inversa só existe se houver outra cotação $(b, a, r')$). Existe no máximo uma cotação por par ordenado.

Uma arbitragem é uma sequência de trocas que começa e termina na mesma moeda e termina com estritamente mais do que se começou (ou seja, o produto das cotações ao longo do ciclo é maior que 1).

Retorne uma lista de moedas $[c_0, c_1, \ldots, c_k]$ com $c_k = c_0$, descrevendo um ciclo de arbitragem, ou a lista vazia `[]` se não existir nenhum. Considere "maior que 1" com tolerância de $10^{-9}$, isto é, o produto do ciclo deve ser maior que $1 + 10^{-9}$ (ciclos com produto 1, como 2.0 seguido de 0.5, não são arbitragem).

Seu algoritmo deve executar em $O(n \cdot m)$, onde $m$ é o número de cotações.

**Restrições:**
- $2 \le n \le 300$, $1 \le m \le 5000$
- $10^{-3} \le r \le 10^{3}$

**Exemplos:**

| Entrada | Saída |
|---|---|
| `n = 3`, `[(0,1,2.0), (1,2,3.0), (2,0,0.2)]` | `[0, 1, 2, 0]` (produto 1.2) |
| `n = 2`, `[(0,1,2.0), (1,0,0.5)]` | `[]` |
| `n = 4`, `[(0,1,1.5), (1,2,0.5), (2,3,2.0)]` | `[]` |

In [ ]:
def question_4(n: int, cotacoes: list) -> list:
    """
    Câmbio Ineficiente.

    n: número de moedas (0..n-1), 2 <= n <= 300.
    cotacoes: lista de tuplas (a, b, r), com a != b: 1 unidade da moeda a vale r
              unidades da moeda b (só nesse sentido). No máximo uma cotação por
              par ordenado.
              1 <= len(cotacoes) <= 5000, 10^-3 <= r <= 10^3.

    Arbitragem: sequência de trocas que começa e termina na mesma moeda com
    estritamente mais do que se começou (produto das cotações do ciclo > 1,
    com tolerância de 1e-9: ciclos de produto 1 NÃO são arbitragem).

    Retorna uma lista [c0, c1, ..., ck] com ck == c0 descrevendo UM ciclo de
    arbitragem, ou [] se não existir nenhum.

    Complexidade exigida: O(n · m), m = len(cotacoes).
    """
    class MoedaNode:
        def __init__(self, id: int) -> None:
            self.id = id
            self.in_edges = list()
            self.out_edges = list()

        def add_in_edge(self, edge:CambioEdge) -> None:
            self.in_edges.append(edge)
            
        def add_out_edge(self, edge:CambioEdge) -> None:
            self.out_edges.append(edge)

    class CambioEdge:
        def __init__(self, origem:MoedaNode, destino:MoedaNode, cambio:float) -> None:
            self.origem = origem
            self.destino = destino
            self.cambio = cambio

            origem.add_out_edge(self)
            destino.add_in_edge(self)

    def explorar_filhos(
            atual_node: MoedaNode,
            pre_order:list[int],
            pre_id:int, 
            pos_order:list[int],
            pos_id:int, 
            acumulado:list[float]
    ) -> tuple[int, list|None, bool]:
        
        for e in atual_node.out_edges:
            pos_id, sol, fechou = avaliar_aresta(e, pre_order, pre_id, pos_order, pos_id, acumulado)
            if sol is not None:
                pos_order[atual_node.id] = pos_id
                pos_id += 1
                if fechou: # ciclo mapeado
                    return pos_id, sol, fechou
                else:
                    sol.append(atual_node)
                    return pos_id, sol, True if atual_node == sol[0] else False

        # nó exaurido, não encontrou solução nas sub árvores filhas
        pos_order[atual_node.id] = pos_id
        return pos_id+1, None, False
        

    def avaliar_aresta(
            atual_edge:CambioEdge, 
            pre_order:list[int],
            pre_id:int, 
            pos_order:list[int],
            pos_id:int, 
            acumulado:list[float]
    ) -> tuple[int, list|None, bool]:
        atual_node = atual_edge.destino

        if (pre_order[atual_node.id] == -1): 
            # o destino ainda não foi explorado
            pre_order[atual_node.id] = pre_id
            acumulado[atual_node.id] = acumulado[atual_edge.origem.id] * atual_edge.cambio
            pre_id += 1
            return explorar_filhos(atual_node, pre_order, pre_id, pos_order, pos_id, acumulado)

        elif (pos_order[atual_node.id] != -1): 
            # destino faz parte de outra sub árvore, logo não gera ciclo
            return pos_id, None, False

        else: 
            # chegou em um ancestral na sub árvore (pois o destino não foi exaurido ainda), logo tem um ciclo aqui
            if (acumulado[atual_edge.origem.id] * atual_edge.cambio <= acumulado[atual_node.id]): 
                # não houve crescimento no ciclo, portanto não é arbitragem
                return pos_id, None, False
            else:
                return pos_id, [atual_node], False

    moedas = [MoedaNode(i) for i in range(n)]
    arestas = [CambioEdge(moedas[a], moedas[b], r) for a, b, r in cotacoes]
    
    pre_order = [-1 for _ in range(n)]
    pos_order = [-1 for _ in range(n)]
    acumulado = [1.0 for _ in range(n)]
    pre_id = pos_id = 0

    for node in moedas:
        if pre_order[node.id] != -1:
            # nó já foi avaliado e não faz parte de um ciclo de arbitragem
            continue

        pre_order[node.id] = pre_id
        pre_id+=1

        _, sol, _ = explorar_filhos(node, pre_order, pre_id, pos_order, pos_id, acumulado)
        if sol is not None:
            return [sol[i-1].id for i in range(len(sol), 0, -1)]

    return list()


In [12]:
def eh_valido(entrada, ciclo, esperado):
    """Qualquer ciclo de arbitragem é aceito; se esperado == [], só [] é aceito."""
    n, cotacoes = entrada
    if esperado == []:
        return ciclo == []
    if not isinstance(ciclo, list) or len(ciclo) < 2 or ciclo[0] != ciclo[-1]:
        return False
    taxas = {(a, b): r for a, b, r in cotacoes}
    produto = 1.0
    for a, b in zip(ciclo, ciclo[1:]):
        if (a, b) not in taxas:
            return False
        produto *= taxas[(a, b)]
    return produto > 1 + 1e-9

entradas = [
    (3, [(0, 1, 2.0), (1, 2, 3.0), (2, 0, 0.2)]),
    (2, [(0, 1, 2.0), (1, 0, 0.5)]),
    (4, [(0, 1, 1.5), (1, 2, 0.5), (2, 3, 2.0)])
]
saidas = [
    "algum ciclo de arbitragem (ex: [0, 1, 2, 0])",
    [],
    []
]
sucessos = 0

print("="*5 + " Exemplos Apresentados " + "="*5 )

for entrada, saida in zip(entradas, saidas):
    print()

    print("Entrada: ", entrada)
    print("Resultado Esperado: ", saida)

    resultado = question_4(*entrada)
    print("Resultado Obtido: ", resultado)

    sucesso = eh_valido(entrada, resultado, saida)
    print("Sucesso: ", sucesso)
    if sucesso: sucessos += 1

    print()
    print("="*34)

print()
print("Total de Sucessos: \t", sucessos)
print("Total de Falhas: \t", len(saidas) - sucessos)

===== Exemplos Apresentados =====

Entrada:  (3, [(0, 1, 2.0), (1, 2, 3.0), (2, 0, 0.2)])
Resultado Esperado:  algum ciclo de arbitragem (ex: [0, 1, 2, 0])
Resultado Obtido:  [0, 1, 2, 0]
Sucesso:  True


Entrada:  (2, [(0, 1, 2.0), (1, 0, 0.5)])
Resultado Esperado:  []
Resultado Obtido:  []
Sucesso:  True


Entrada:  (4, [(0, 1, 1.5), (1, 2, 0.5), (2, 3, 2.0)])
Resultado Esperado:  []
Resultado Obtido:  []
Sucesso:  True


Total de Sucessos: 	 3
Total de Falhas: 	 0


---
## 5. Organograma da Empresa

Uma empresa tem $n$ funcionários numerados de $0$ a $n-1$, organizados em uma hierarquia com raiz no funcionário $0$. A hierarquia é dada como uma lista de $n-1$ pares $(u, v)$ que indicam que $u$ e $v$ estão diretamente ligados (sem indicar quem é o chefe de quem; a raiz é sempre o 0).

O RH fará $q$ consultas, cada uma da forma `(tipo, u, v)`:

- `("A", u, v)`: $u$ é chefe (direto ou indireto) de $v$?
- `("P", u, v)`: $u$ e $v$ são primos, isto é, estão no mesmo nível da hierarquia mas têm chefes diretos diferentes?

Em ambas as consultas, se $u = v$ a resposta é `False`.

Retorne a lista com as respostas (booleanos) na ordem das consultas, com tempo total $O(n + q)$: cada consulta, depois de um pré-processamento, deve custar $O(1)$.

**Restrições:**
- $2 \le n \le 2 \cdot 10^5$, $1 \le q \le 2 \cdot 10^5$
- A hierarquia pode ser uma "cadeia" de $2 \cdot 10^5$ funcionários.

**Exemplo**, com $n = 7$ e `ligacoes = [(0,1), (0,2), (1,3), (1,4), (2,5), (2,6)]`:

| Consulta | Saída | Motivo |
|---|---|---|
| `("A", 0, 5)` | `True` | 0 é chefe indireto de 5 (0 → 2 → 5) |
| `("A", 1, 5)` | `False` | 5 está sob o 2, não sob o 1 |
| `("A", 3, 3)` | `False` | $u = v$ |
| `("P", 3, 5)` | `True` | mesmo nível, chefes diretos 1 e 2 |
| `("P", 3, 4)` | `False` | mesmo chefe direto (1) |
| `("P", 1, 3)` | `False` | níveis diferentes |

In [ ]:
def question_5(n: int, ligacoes: list, consultas: list) -> list:
    """
    Organograma da Empresa.

    n: número de funcionários (0..n-1), raiz da hierarquia é o 0.
       2 <= n <= 2 * 10^5. A hierarquia pode ser uma "cadeia".
    ligacoes: n-1 pares (u, v) que indicam que u e v estão diretamente ligados
              (não indicam quem é o chefe de quem).
    consultas: lista de q tuplas (tipo, u, v), 1 <= q <= 2 * 10^5:
        ("A", u, v): u é chefe (direto ou indireto) de v?
        ("P", u, v): u e v são primos (mesmo nível, chefes diretos diferentes)?
    Em ambas as consultas, se u == v a resposta é False.

    Retorna a lista de respostas (bool) na ordem das consultas.

    Complexidade exigida: O(n + q) no total (cada consulta O(1) após o
    pré-processamento).
    """
    pass

In [ ]:
entradas = [
    (7,
     [(0, 1), (0, 2), (1, 3), (1, 4), (2, 5), (2, 6)],
     [("A", 0, 5), ("A", 1, 5), ("A", 3, 3), ("P", 3, 5), ("P", 3, 4), ("P", 1, 3)])
]
saidas = [
    [True, False, False, True, False, False]
]
sucessos = 0

print("="*5 + " Exemplos Apresentados " + "="*5 )

for entrada, saida in zip(entradas, saidas):
    print()

    print("Entrada: ", entrada)
    print("Resultado Esperado: ", saida)

    resultado = question_5(*entrada)
    print("Resultado Obtido: ", resultado)

    sucesso = resultado == saida
    print("Sucesso: ", sucesso)
    if sucesso: sucessos += 1

    print()
    print("="*34)

print()
print("Total de Sucessos: \t", sucessos)
print("Total de Falhas: \t", len(saidas) - sucessos)

---
## 6. Abastecimento de Água

Uma vila tem $n$ casas, numeradas de $0$ a $n-1$. Para abastecer uma casa com água, há duas opções:

- cavar um poço na própria casa, com custo `poço[i]`;
- ligá-la, por um cano, a outra casa que já seja abastecida. Há uma lista de canos possíveis $(i, j, \text{custo})$, com $i \neq j$ e no máximo um cano por par de casas, que funcionam nos dois sentidos, e a água é transmitida por toda sequência de canos ligados.

Retorne o menor custo total para abastecer todas as casas.

Seu algoritmo deve executar em $O((n + m) \log n)$, onde $m$ é o número de canos possíveis.

**Restrições:**
- $1 \le n \le 10^4$, $0 \le m \le 10^5$
- $1 \le \text{custos} \le 10^5$

**Exemplos:**

| Entrada | Saída |
|---|---|
| `poço = [5,4,6,10]`, `canos = [(0,1,1),(1,2,2),(2,3,3),(0,3,8)]` | 10 |
| `poço = [1,2,2]`, `canos = [(0,1,1),(1,2,1)]` | 3 |

In [ ]:
def question_6(pocos: list, canos: list) -> int:
    """
    Abastecimento de Água.

    pocos: lista de n custos; pocos[i] é o custo de cavar um poço na casa i.
    canos: lista de tuplas (i, j, custo), com i != j e no máximo um cano por par
           de casas: cano possível entre as casas i e j (funciona nos dois
           sentidos; a água é transmitida por toda sequência de canos ligados).
    1 <= n <= 10^4, 0 <= len(canos) <= 10^5, 1 <= custos <= 10^5.

    Retorna o menor custo total para abastecer todas as casas.

    Complexidade exigida: O((n + m) log n), m = len(canos).
    """
    pass

In [ ]:
entradas = [
    ([5, 4, 6, 10], [(0, 1, 1), (1, 2, 2), (2, 3, 3), (0, 3, 8)]),
    ([1, 2, 2], [(0, 1, 1), (1, 2, 1)])
]
saidas = [
    10,
    3
]
sucessos = 0

print("="*5 + " Exemplos Apresentados " + "="*5 )

for entrada, saida in zip(entradas, saidas):
    print()

    print("Entrada: ", entrada)
    print("Resultado Esperado: ", saida)

    resultado = question_6(*entrada)
    print("Resultado Obtido: ", resultado)

    sucesso = resultado == saida
    print("Sucesso: ", sucesso)
    if sucesso: sucessos += 1

    print()
    print("="*34)

print()
print("Total de Sucessos: \t", sucessos)
print("Total de Falhas: \t", len(saidas) - sucessos)

---
## 7. Trilha de Escalada

Um mapa de relevo é uma matriz $n \times m$ de altitudes. Uma escalada começa em qualquer célula e, a cada passo, move-se para uma das quatro células vizinhas (cima, baixo, esquerda, direita) cuja altitude seja **estritamente maior** que a da célula atual. Uma escalada pode ter uma única célula.

Retorne uma tupla `(L, c)`, em que $L$ é o maior número de células de uma escalada e $c$ é a quantidade de escaladas distintas com exatamente $L$ células, módulo $10^9 + 7$. (Duas escaladas são distintas se diferem na sequência de células visitadas.)

Seu algoritmo deve executar em $O(n \cdot m)$ ou $O(n \cdot m \log(n \cdot m))$.

**Restrições:**
- $1 \le n, m \le 300$
- $0 \le \text{altitude} \le 10^9$ (valores podem se repetir)

**Exemplos:**

| Entrada | Saída |
|---|---|
| `[[1,2,3],[2,3,4],[3,4,5]]` | `(5, 6)` |
| `[[7,7],[7,7]]` | `(1, 4)` |
| `[[1,3],[2,1]]` | `(2, 4)` |
| `[[3,4,5],[3,2,6],[2,2,1]]` | `(4, 2)` |

In [ ]:
def question_7(relevo: list) -> tuple:
    """
    Trilha de Escalada.

    relevo: matriz n x m (lista de listas) de altitudes,
            1 <= n, m <= 300, 0 <= altitude <= 10^9 (valores podem se repetir).
    Uma escalada começa em qualquer célula e a cada passo move-se para uma das
    4 vizinhas de altitude ESTRITAMENTE maior. Pode ter uma única célula.

    Retorna (L, c): L é o maior número de células de uma escalada e c é a
    quantidade de escaladas distintas com exatamente L células, módulo 10^9 + 7.
    (Duas escaladas são distintas se diferem na sequência de células visitadas.)

    Complexidade exigida: O(n · m) ou O(n · m · log(n · m)).
    """
    pass

In [ ]:
entradas = [
    ([[1, 2, 3], [2, 3, 4], [3, 4, 5]],),
    ([[7, 7], [7, 7]],),
    ([[1, 3], [2, 1]],),
    ([[3, 4, 5], [3, 2, 6], [2, 2, 1]],)
]
saidas = [
    (5, 6),
    (1, 4),
    (2, 4),
    (4, 2)
]
sucessos = 0

print("="*5 + " Exemplos Apresentados " + "="*5 )

for entrada, saida in zip(entradas, saidas):
    print()

    print("Entrada: ", entrada)
    print("Resultado Esperado: ", saida)

    resultado = question_7(*entrada)
    print("Resultado Obtido: ", resultado)

    sucesso = resultado == saida
    print("Sucesso: ", sucesso)
    if sucesso: sucessos += 1

    print()
    print("="*34)

print()
print("Total de Sucessos: \t", sucessos)
print("Total de Falhas: \t", len(saidas) - sucessos)

---
## 8. O Carteiro Preguiçoso

Uma região tem $n$ povoados, numerados de $0$ a $n-1$, ligados por estradas de mão dupla (lista de pares $(u, v)$); é possível chegar de qualquer povoado a qualquer outro. Um carteiro precisa passar por todos os povoados pelo menos uma vez, pode começar e terminar onde quiser, e pode passar várias vezes pelo mesmo povoado ou estrada.

**a)** (Código) Para $n \le 12$, retorne o menor número de estradas percorridas (contando repetições) em um trajeto que visita todos os povoados. Qual a complexidade do seu algoritmo em função de $n$ e do número de estradas?

**b)** Mostre que, para um dado $n$, existe um trajeto que visita todos os povoados percorrendo exatamente $n-1$ estradas se e somente se a região possui um caminho hamiltoniano (caminho simples que passa por todos os vértices). O que isso diz sobre a existência de um algoritmo polinomial para o problema do item (a) com $n$ grande? Justifique usando os conceitos de classes de complexidade e redução polinomial.

**c)** Se a região fosse uma árvore (sem ciclos), o problema passaria a ter algoritmo polinomial? Se sim, descreva um algoritmo $O(n)$. Por que isso não contradiz o item (b)?

**Restrições (item a):**
- $2 \le n \le 12$
- O grafo é conexo.

**Exemplos (item a):**

| Povoados e estradas | Saída |
|---|---|
| `n = 3`, `[(0,1),(1,2)]` | 2 |
| `n = 4`, `[(0,1),(0,2),(0,3)]` | 4 |
| `n = 4`, `[(0,1),(1,2),(2,3),(3,0)]` | 3 |
| `n = 5`, `[(0,1),(0,2),(0,3),(0,4)]` | 6 |

In [ ]:
def question_8_a(n: int, estradas: list) -> int:
    """
    O Carteiro Preguiçoso — item (a).

    n: número de povoados (0..n-1), 2 <= n <= 12.
    estradas: lista de pares (u, v) (mão dupla); o grafo é conexo.
    O carteiro passa por todos os povoados pelo menos uma vez, começa e termina
    onde quiser e pode repetir povoados e estradas.

    Retorna o menor número de estradas percorridas (contando repetições).

    RESPONDA AQUI (substitua este texto):
    Complexidade do seu algoritmo em função de n e do número de estradas: ...
    """
    pass

In [ ]:
entradas = [
    (3, [(0, 1), (1, 2)]),
    (4, [(0, 1), (0, 2), (0, 3)]),
    (4, [(0, 1), (1, 2), (2, 3), (3, 0)]),
    (5, [(0, 1), (0, 2), (0, 3), (0, 4)])
]
saidas = [
    2,
    4,
    3,
    6
]
sucessos = 0

print("="*5 + " Exemplos Apresentados " + "="*5 )

for entrada, saida in zip(entradas, saidas):
    print()

    print("Entrada: ", entrada)
    print("Resultado Esperado: ", saida)

    resultado = question_8_a(*entrada)
    print("Resultado Obtido: ", resultado)

    sucesso = resultado == saida
    print("Sucesso: ", sucesso)
    if sucesso: sucessos += 1

    print()
    print("="*34)

print()
print("Total de Sucessos: \t", sucessos)
print("Total de Falhas: \t", len(saidas) - sucessos)

In [ ]:
def question_8_b():
    """
    O Carteiro Preguiçoso — item (b) (discursivo, não é corrigido
    automaticamente).

    RESPONDA AQUI (substitua este texto):
    (1) Prova de que existe trajeto com exatamente n-1 estradas se e somente se
        existe caminho hamiltoniano: ...
    (2) O que isso diz sobre um algoritmo polinomial para o item (a) com n
        grande (classes de complexidade e redução polinomial): ...
    """
    pass

In [ ]:
def question_8_c():
    """
    O Carteiro Preguiçoso — item (c) (discursivo, não é corrigido
    automaticamente).

    RESPONDA AQUI (substitua este texto):
    (1) Se a região fosse uma árvore, o problema teria algoritmo polinomial?
        Descreva um algoritmo O(n): ...
    (2) Por que isso não contradiz o item (b)? ...
    """
    pass